# 5色ボール：坂と正面アームによる左右仕分け（3セット）

**提出物の種類：3D動作アニメーションの再構成。AI推論結果や剛体力学の検証動画ではありません。**

大きな赤・青・黄・緑・黒の球が奥の坂から手前へ下り、正面の二関節アームが左右のトレーへ送る映像です。承認された構図と最終動画の動作を再現します。氏名・所属・メールアドレスは記載していません。

## 完成動画の仕様

| 項目 | 値 |
|---|---|
| 球・セット | 5色×3セット＝15球 |
| 右トレー | 赤・青 |
| 左トレー | 黄・緑・黒（破砕表現なし） |
| 出力 | MP4 / H.264 / 1280×720 / 30 fps |
| 長さ | 4.8秒×15球＝72秒、2,160フレーム |
| カメラ | 位置(3.8, −7.2, 5.3)、注視点(0, 0.82, 0.44)、画角38度 |
| 描画 | PyBulletのCPU版TinyRenderer、影なし |

黒球も左右に振り分ける仕様を優先しました。色ごとの左右ルールと出現順はコードで明示的に指定しています。


## 知恵を絞った点：見た目だけで動作が伝わる設計

### 1. 遠い俯瞰ではなく、坂・球・アームを一度に見せる
最初の出力は平らな装置を遠くから見た映像で、球が小さく、求められた「坂を下って正面のアームへ来る」動作が伝わりませんでした。画角だけの変更ではなく、傾斜路、支柱、左右トレー、手前の二関節アームを新たに組みました。

球の半径は試作の0.32から0.42へ拡大し、画角を43度から38度へ絞りました。坂の上端と左右トレーを画面内に残しながら、球を完成映像でおおむね80〜100画素の大きさに見せています。球には滑らかなメッシュを使用し、小さな表面マークで回転の変化も見えるようにしました。

### 2. 二関節アームを逆運動学で組み立てる
肩位置を S=(0, −1.65)、先端位置を T=(x,y)、リンク長を L₁=1.15、L₂=1.65 とします。距離 d=||T−S||、方向角 φ=atan2(Tᵧ−Sᵧ,Tₓ−Sₓ) を求め、余弦定理から

`α = acos(clip((L₁² + d² − L₂²) / (2 L₁ d), −1, 1))`

と置きます。肘位置は `E = S + L₁(cos(φ−α), sin(φ−α))` です。肩→肘、肘→先端をそれぞれ円柱で結び、関節とパドルを描きます。単に箱を横移動させるのではなく、先端の位置に応じて二つのリンクが折れ曲がるため、ロボットアームとして認識しやすくなります。

`clip` は浮動小数点の丸めで acos の定義域から外れることを防ぐためです。ただし、これは到達不能な目標を解決する機能ではありません。設定した軌道はリンク長の範囲内です。

### 3. 時刻から決める動作を5段階に分ける
フレーム番号 n から `t=n/30` を求めます。球番号は `floor(t/4.8)`、セット番号はその球番号を5で割った整数部分です。この方法で毎回同じ順序・長さになります。

| 球ごとの局所時刻 | 動作 |
|---|---|
| 0〜1.65秒 | 坂を下る |
| 1.65〜2.20秒 | 「ボールの色は？」を表示 |
| 2.20〜2.75秒 | 色の回答表示・アームを球の横へ移動 |
| 2.75〜4.00秒 | 左右へ横送り |
| 4.00〜4.80秒 | 球をトレー位置に保持し、アームを戻す |

坂下りでは `u=t/1.65` とし、`y=3.02−3.12u²` で奥から手前へ加速する見え方を作ります。高さは傾斜0.5の路面と球半径に合わせ、坂の出口近くでは平らなデッキの高さへつなぎます。**重力・摩擦を積分して得た運動ではなく、意図して設計した軌道です。**

アームの移動には `s(u)=u²(3−2u)` を使います。`s(0)=0`、`s(1)=1`、両端の微分が0なので、線形補間より開始・停止が穏やかになります。

### 4. アームが球を突き抜ける見え方を避ける
原点から球の反対側へ直線で動かすと、接近中にパドルが球を横切る配置になり得ます。そのため、まずカメラ側の安全なY位置へ移し、その後に球の横へ進む二段階の経路にしました。

横送り中は球中心を `パドルX + 向き×(球半径+0.12)` に置きます。パドルの半幅0.115との差により、見た目の隙間は約0.005となります。トレーの内側の壁は開け、球が壁を通過して見える問題も避けました。

**この位置関係は視覚的な接触を表すための配置であり、衝突検出や接触力の測定ではありません。**

### 5. 読める日本語字幕と、見失わない進捗表示
Noto Sans JPの可変フォントを太さ600で使用しています。小さく細い文字だった試作から太字へ変更しました。タイトル、セット数、球番号、現在の動作を分けて表示し、球やアームに重ねない位置に置きました。

### 6. 画質だけでなく再現性と提出安全性を考える
ソース全体をNotebookに内包し、外部フォントは公式配布元の特定コミットとSHA-256で固定しました。フォントバイナリには第三者の氏名がメタデータとして含まれるため、提出物には同梱せず、実行時に取得します。生成動画は提出前にフレーム数・寸法・長さ・画面内文字・メタデータを確認します。


## 実際に苦労した点と、行った対応

以下は作業で実際に生じた問題です。経験していない試行や実験結果を付け加えていません。

| 問題 | 実際の対応・分かったこと |
|---|---|
| 先の動画が意図と違った | 先の平面・小球・遠い俯瞰を不採用とし、坂・大型球・正面の二関節アームに作り直しました。途中の静止画で構図を確認してから全編を出力しました。 |
| Python 3.12でPyBullet 3.2.7の適合wheelが見つからなかった | 公式PyPIのソース配布からビルドしました。別シミュレーターへの置換ではなく、描画環境を整えました。 |
| 既定のC++コンパイラが存在しなかった | 利用可能なgcc/g++を明示しました。OS設定や権限制限の回避は行っていません。 |
| C APIのポインタ型警告がコンパイルエラーになった | プロジェクトの許容範囲にあるNumPy 1.26.4を使い、`-Wno-error=incompatible-pointer-types`で当該警告をエラー扱いしない設定を加え、ビルドできました。警告の原因が修正されたと主張するものではありません。 |
| 球と字幕が目立たなかった | 球径とカメラ画角を調整し、細い文字を太字へ変更しました。球は滑らかなメッシュに替えました。 |
| アームの接近経路とトレー入口が不自然だった | 接近を二段階に分け、トレーの内側の壁をなくしました。視覚上の通過経路を明確にしました。 |
| 影を加えた試作で大きな不自然な影が出た | 影付きのプレビューを比較した結果、承認された最終版では影を使わず、形状と色の読みやすさを優先しました。 |
| HDのCPUレンダリングに時間がかかった | 72秒を2,160枚の画像として生成するため、この制作環境で全編に約20分かかりました。環境依存であり、実時間動作の性能評価ではありません。先に少数フレームで構図を確認し、全編のやり直しを減らしました。 |
| 提出ソースに実行環境の絶対パスが残っていた | フォント参照をプロジェクト内の相対パスへ変更し、ログ・Library識別子・作業用ファイルを提出対象から外しました。 |

## 限界と誤解してはいけない点

- 球の色は `COLORS` / `ORDERS` から直接使います。画像認識や学習済み分類器で推定していません。「答え」は表示用の既知ラベルです。
- PyBulletを使っていますが、本版は視覚モデルを時刻ごとの座標に配置するアニメーションです。重力、摩擦、トルク制御、衝突力を検証した物理シミュレーションではありません。
- 球の回転マークは回転の見え方を示します。滑りなし接触の厳密な力学を保証しません。
- 横送り終端からトレー内の固定位置へ移る際、球中心を約0.10だけ調整する箇所があります。承認された動画と同じ動作を保つため、そのまま記録しています。連続力学として厳密とは言えません。
- 次の球の開始時には前の球を消します。トレー内への15球の積み重ねは表現していません。
- 過去動画の完全復元ではなく、求められた構図を作り直した映像です。今回の最終動画と同じコード上の動作は再現できますが、OS・フォント描画・FFmpeg等の違いによりMP4のバイト単位の一致までは保証しません。


## 再現手順

1. このNotebookをPython環境またはGoogle Colabで開きます。ネットワーク接続が必要です。
2. 依存ライブラリを導入します。Python 3.12ではソースビルドになる場合があり、gcc/g++等のコンパイラが必要です。元の制作時はPython 3.12、NumPy 1.26.4、PyBullet 3.2.7で実行しました。
3. ソース展開セルとフォント取得セルを実行します。フォントとOFLライセンスは公式配布元からダウンロードし、ハッシュが一致しなければ停止します。
4. 初期設定ではプレビュー6枚だけを出力します。構図を確認します。
5. 全編が必要なときだけ `RUN_FULL_RENDER = True` に変更してレンダリングセルを実行します。
6. 最後の検証セルでMP4の72秒・2,160フレーム・HDを確認します。

公開リポジトリを使う場合は `pip install -r requirements.txt`、`python setup_font.py`、`python render.py --preview`、`python render.py` の順でも再現できます。Python 3.12でビルド問題が生じる場合はNotebookの依存導入セルと同じ環境変数を設定してください。

`config.json` は仕様を確認する一覧です。レンダラーが自動読込みする設定ファイルではありません。動作を変更する場合は `render.py` 内の対応する定数・座標を編集してください。

ライブラリ取得は公式PyPI、フォント取得は公式Google Fontsリポジトリを使用します。ネットワークや実行権限が許可されない環境では、それらを回避せず処理を止めてください。


In [ ]:
# 1. 依存ライブラリ（既に導入済みなら False）
INSTALL_DEPENDENCIES = True
import os, sys, subprocess
from pathlib import Path
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "--index-url", "https://pypi.org/simple",
                    "numpy==1.26.4", "Pillow==11.3.0", "imageio==2.38.1", "imageio-ffmpeg==0.7.0"], check=True)
    build_env = os.environ.copy()
    build_env.update({"CC": "gcc", "CXX": "g++", "CFLAGS": "-Wno-error=incompatible-pointer-types"})
    subprocess.run([sys.executable, "-m", "pip", "install", "--index-url", "https://pypi.org/simple",
                    "pybullet==3.2.7"], check=True, env=build_env)
print("依存ライブラリの準備が完了しました。")


## ソース全体の展開
次のセルには省略のないレンダラーが入っています。`physical_ai_work` に相対パスで保存します。メインの設定、逆運動学、時刻別動作、描画、MP4出力をこのセルで確認できます。


In [ ]:
WORK = Path('physical_ai_work')
WORK.mkdir(exist_ok=True)
SOURCE = r'''"""5色ボール・坂・正面アーム：承認構図の再構成アニメーション。
学習済み分類器・力学シミュレーションではなく、時刻から姿勢を定める映像。
実行: python render.py --preview / python render.py
"""
from pathlib import Path
import math, argparse
import numpy as np
import pybullet as p
import pybullet_data
from PIL import Image, ImageDraw, ImageFont
import imageio.v2 as imageio
ROOT = Path(__file__).parent
W, H, FPS = (1280, 720, 30)
R = 0.42
DURATION = 4.8
COLORS = {'red': (0.94, 0.055, 0.045, 1), 'blue': (0.025, 0.24, 0.98, 1), 'yellow': (1, 0.76, 0.025, 1), 'green': (0.035, 0.68, 0.22, 1), 'black': (0.035, 0.045, 0.065, 1)}
JP = {'red': '赤', 'blue': '青', 'yellow': '黄', 'green': '緑', 'black': '黒'}
DIR = {'red': 1, 'blue': 1, 'yellow': -1, 'green': -1, 'black': -1}
ORDERS = [['red', 'blue', 'yellow', 'green', 'black'], ['green', 'red', 'black', 'blue', 'yellow'], ['blue', 'yellow', 'green', 'black', 'red']]
font_path = str(ROOT / 'assets' / 'NotoSansJP-VF.ttf')
fonts = {n: ImageFont.truetype(font_path, n) for n in [20, 24, 28, 34]}
for font in fonts.values():
    font.set_variation_by_axes([600])

def box(half, pos, color, orn=None):
    v = p.createVisualShape(p.GEOM_BOX, halfExtents=half, rgbaColor=color)
    return p.createMultiBody(baseMass=0, baseVisualShapeIndex=v, basePosition=pos, baseOrientation=orn or [0, 0, 0, 1])
sphere_shapes = {}

def sphere(radius, pos, color):
    key = (radius, tuple(color))
    if key not in sphere_shapes:
        sphere_shapes[key] = p.createVisualShape(p.GEOM_MESH, fileName=str(Path(pybullet_data.getDataPath()) / 'sphere_smooth.obj'), meshScale=[radius] * 3, rgbaColor=color) if radius > 0.1 else p.createVisualShape(p.GEOM_SPHERE, radius=radius, rgbaColor=color)
    return p.createMultiBody(baseMass=0, baseVisualShapeIndex=sphere_shapes[key], basePosition=pos)

def cylinder(radius, length, pos, color):
    v = p.createVisualShape(p.GEOM_CYLINDER, radius=radius, length=length, rgbaColor=color)
    return p.createMultiBody(baseMass=0, baseVisualShapeIndex=v, basePosition=pos)

def link(a, b, r, color):
    a, b = (np.array(a), np.array(b))
    d = b - a
    length = np.linalg.norm(d)
    z = d / length
    axis = np.cross([0, 0, 1], z)
    angle = math.acos(np.clip(z[2], -1, 1))
    q = p.getQuaternionFromAxisAngle(axis.tolist(), angle) if np.linalg.norm(axis) > 1e-06 else [0, 0, 0, 1]
    v = p.createVisualShape(p.GEOM_CYLINDER, radius=r, length=length, rgbaColor=color)
    return p.createMultiBody(baseMass=0, baseVisualShapeIndex=v, basePosition=((a + b) / 2).tolist(), baseOrientation=q)

def smooth(u):
    u = np.clip(u, 0, 1)
    return float(u * u * (3 - 2 * u))
p.connect(p.DIRECT)
box([100, 100, 0.05], [0, 0, -0.3], [0.91, 0.93, 0.96, 1])
box([3.0, 2.9, 0.1], [0, 0.65, -0.11], [0.68, 0.74, 0.8, 1])
theta = math.atan(0.5)
q = p.getQuaternionFromEuler([theta, 0, 0])
box([0.65, 1.7, 0.07], [0, 1.65, 0.825], [0.4, 0.53, 0.65, 1], q)
for x in [-0.72, 0.72]:
    box([0.055, 1.73, 0.15], [x, 1.65, 1.02], [0.8, 0.87, 0.93, 1], q)
for y in [1.05, 2.15, 3.1]:
    box([0.51, 0.06, max(0.05, y * 0.25)], [0, y, y * 0.25 - 0.09], [0.36, 0.43, 0.5, 1])
box([1.22, 0.65, 0.035], [0, -0.18, 0.01], [0.49, 0.61, 0.7, 1])
for x, c in [(-2.1, [0.08, 0.48, 0.52, 1]), (2.1, [0.17, 0.36, 0.64, 1])]:
    box([0.63, 0.75, 0.04], [x, -0.05, -0.005], c)
    for dx in [-0.65] if x < 0 else [0.65]:
        box([0.035, 0.75, 0.12], [x + dx, -0.05, 0.1], c)
    box([0.65, 0.035, 0.12], [x, 0.71, 0.1], c)
    box([0.65, 0.035, 0.07], [x, -0.81, 0.055], c)
cylinder(0.31, 0.32, [0, -1.65, 0.18], [0.13, 0.19, 0.25, 1])
cylinder(0.23, 0.15, [0, -1.65, 0.4], [0.95, 0.49, 0.08, 1])
view = p.computeViewMatrix([3.8, -7.2, 5.3], [0, 0.82, 0.44], [0, 0, 1])
proj = p.computeProjectionMatrixFOV(38, W / H, 0.1, 30)

def arm(end):
    shoulder = np.array([0.0, -1.65, 0.58])
    target = np.array([end[0], end[1], 0.58])
    d = target[:2] - shoulder[:2]
    l1, l2 = (1.15, 1.65)
    dist = np.linalg.norm(d)
    phi = math.atan2(d[1], d[0])
    alpha = math.acos(np.clip((l1 * l1 + dist * dist - l2 * l2) / (2 * l1 * dist), -1, 1))
    elbow = shoulder + np.array([l1 * math.cos(phi - alpha), l1 * math.sin(phi - alpha), 0])
    ids = [link(shoulder, elbow, 0.125, [0.98, 0.56, 0.11, 1]), link(elbow, target, 0.105, [1, 0.65, 0.16, 1])]
    for c in [shoulder, elbow, target]:
        ids.append(cylinder(0.17, 0.14, c, [0.14, 0.2, 0.26, 1]))
    ids.append(box([0.1, 0.37, 0.28], [end[0], end[1], 0.36], [0.95, 0.57, 0.13, 1]))
    ids.append(box([0.115, 0.34, 0.2], [end[0], end[1], 0.34], [0.2, 0.25, 0.29, 1]))
    return ids

def state(t, color):
    direction = DIR[color]
    home = np.array([0.8, -1.2])
    start = np.array([-direction * (R + 0.12), -0.1])
    end = np.array([direction * 1.55, -0.1])
    if t < 1.65:
        u = t / 1.65
        y = 3.02 - 3.12 * u * u
        z = R + 0.05 + max(0, (y + 0.1) / 0.3) * (0.1 + 0.078 + R * math.sqrt(1.25) - R - 0.05) if y < 0.2 else 0.5 * y + 0.078 + R * math.sqrt(1.25)
        pos = [0, y, z]
        target = home
        phase = '坂を下っています'
    elif t < 2.2:
        pos = [0, -0.1, R + 0.05]
        target = home
        phase = 'ボールの色は？'
    elif t < 2.75:
        pos = [0, -0.1, R + 0.05]
        safe = np.array([start[0], -0.95])
        target = home + (safe - home) * smooth((t - 2.2) / 0.275) if t < 2.475 else safe + (start - safe) * smooth((t - 2.475) / 0.275)
        phase = '答え：' + JP[color]
    elif t < 4.0:
        u = smooth((t - 2.75) / 1.25)
        target = start + (end - start) * u
        pos = [target[0] + direction * (R + 0.12), -0.1, R + 0.05]
        phase = ('右へ' if direction > 0 else '左へ') + '仕分け'
    else:
        pos = [direction * 1.99, -0.1, R + 0.05]
        target = end + (home - end) * smooth((t - 4) / 0.8)
        phase = '仕分け完了'
    return (pos, target, phase)

def frame(global_t):
    idx = min(14, int(global_t / DURATION))
    t = global_t - idx * DURATION
    setno = idx // 5
    color = ORDERS[setno][idx % 5]
    pos, target, phase = state(t, color)
    ids = arm(target)
    ids.append(sphere(R, pos, COLORS[color]))
    q = p.getQuaternionFromEuler([(3.02 - pos[1]) / R, pos[0] / R, 0])
    p.resetBasePositionAndOrientation(ids[-1], pos, q)
    rot = np.asarray(p.getMatrixFromQuaternion(q)).reshape(3, 3)
    for local in [[R * 0.98, 0, 0], [0, R * 0.98, 0], [0, 0, R * 0.98]]:
        marker = np.array(pos) + rot @ np.array(local)
        ids.append(sphere(0.026, marker.tolist(), [0.85, 0.9, 0.94, 1]))
    raw = p.getCameraImage(W, H, view, proj, renderer=p.ER_TINY_RENDERER, shadow=0, lightDirection=[-3, -4, 8], lightAmbientCoeff=0.65, lightDiffuseCoeff=0.65)
    img = Image.fromarray(np.asarray(raw[2], dtype=np.uint8).reshape(H, W, 4)[:, :, :3])
    d = ImageDraw.Draw(img)
    d.rectangle([0, 0, W, 94], fill='#122434')
    d.text((36, 12), '5色ボールの仕分け', font=fonts[34], fill='white')
    d.text((37, 55), '坂を下るボールを、正面のアームで左右へ', font=fonts[20], fill='#c3d3df')
    d.rounded_rectangle((1010, 20, 1244, 74), radius=16, fill='#284556')
    d.text((1030, 28), f'{setno + 1} / 3 セット · {idx % 5 + 1}/5', font=fonts[24], fill='white')
    d.rounded_rectangle((388, 638, 892, 704), radius=18, fill='#ffffff', outline='#bacbd7', width=2)
    bb = d.textbbox((0, 0), phase, font=fonts[28])
    d.text(((W - (bb[2] - bb[0])) / 2, 649), phase, font=fonts[28], fill='#172d3e')
    d.text((34, 664), '左 ← 黄・緑・黒', font=fonts[24], fill='#164d50')
    d.text((1000, 664), '赤・青 → 右', font=fonts[24], fill='#224674')
    for i in ids:
        p.removeBody(i)
    return np.asarray(img)
if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--preview', action='store_true')
    args = ap.parse_args()
    if args.preview:
        times = [0.35, 1.2, 2.35, 3.35, 4.35, 12.95]
        canvas = Image.new('RGB', (W * 2, H * 3))
        for i, t in enumerate(times):
            im = Image.fromarray(frame(t))
            im.save(ROOT / f'preview_{i}.png')
            canvas.paste(im, (i % 2 * W, i // 2 * H))
        canvas.save(ROOT / 'preview_sheet.jpg')
        print('preview ready')
    else:
        writer = imageio.get_writer(ROOT / 'PhysicalAI_5色ボール_坂・正面アーム左右仕分け_3セット_20261101.mp4', fps=FPS, codec='libx264', quality=8, macro_block_size=1)
        for i in range(round(15 * DURATION * FPS)):
            writer.append_data(frame(i / FPS))
            if i % 144 == 0:
                print(f'{i}/{15 * DURATION * FPS}', flush=True)
        writer.close()
p.disconnect()
'''
(WORK / 'render.py').write_text(SOURCE, encoding='utf-8')
print('レンダラーを展開しました。')


In [ ]:
# 公式フォント取得コード。個人パスや認証情報は不要です。
FONT_SETUP = r'''"""公式フォントを実行時に取得し、ハッシュとライセンスを確認する。"""
from pathlib import Path
import hashlib
import urllib.request

ROOT = Path(__file__).resolve().parent
COMMIT = "295d98a7a0c17c68f1341eaeea354e7960ea70d3"
BASE = f"https://raw.githubusercontent.com/google/fonts/{COMMIT}/ofl/notosansjp/"
FILES = {
    "NotoSansJP-VF.ttf": ("NotoSansJP%5Bwght%5D.ttf", "c2f3b4d463500a2ddcd3849cded1fceeb9fd6d1c32e6cbecd568453ba50fc68f"),
    "OFL.txt": ("OFL.txt", "1c05c68c34f9708415aada51f17e1b0092d2cea709bf4a94cd38114f9e73d7d9"),
}

def main():
    assets = ROOT / "assets"
    assets.mkdir(exist_ok=True)
    for name, (remote, expected) in FILES.items():
        path = assets / name
        data = path.read_bytes() if path.is_file() else urllib.request.urlopen(BASE + remote, timeout=90).read()
        if hashlib.sha256(data).hexdigest() != expected:
            raise RuntimeError("公式配布ファイルのハッシュが一致しません。処理を中止します。")
        path.write_bytes(data)
    print("フォントとライセンスのハッシュを確認しました。")

if __name__ == "__main__":
    main()
'''
(WORK / 'setup_font.py').write_text(FONT_SETUP, encoding='utf-8')
subprocess.run([sys.executable, 'setup_font.py'], cwd=WORK, check=True)


In [ ]:
# 仕様一覧（自動読み込み設定ではありません）
import json
CONFIG = json.loads('{"width": 1280, "height": 720, "fps": 30, "sets": 3, "balls_per_set": 5, "ball_radius": 0.42, "seconds_per_ball": 4.8, "duration_seconds": 72, "frames": 2160, "camera_position": [3.8, -7.2, 5.3], "camera_target": [0, 0.82, 0.44], "fov_degrees": 38, "arm_lengths": [1.15, 1.65], "orders": [["red", "blue", "yellow", "green", "black"], ["green", "red", "black", "blue", "yellow"], ["blue", "yellow", "green", "black", "red"]], "right": ["red", "blue"], "left": ["yellow", "green", "black"], "font_weight": 600, "renderer": "PyBullet TinyRenderer CPU", "control": "kinematic animation; no trained classifier; no dynamic contact solver", "configuration_note": "このJSONは仕様一覧です。変更する場合は render.py 内の対応する定数・座標も変更してください。"}')
(WORK / 'config.json').write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')
CONFIG


In [ ]:
# 2. まず少数フレームだけ生成
subprocess.run([sys.executable, 'render.py', '--preview'], cwd=WORK, check=True)
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(WORK / 'preview_sheet.jpg'), width=1000))


In [ ]:
# 3. 全編出力は明示的に True にした場合のみ実行
RUN_FULL_RENDER = False
if RUN_FULL_RENDER:
    subprocess.run([sys.executable, 'render.py'], cwd=WORK, check=True)
else:
    print('プレビューのみ。72秒の全編を作る場合は RUN_FULL_RENDER = True にしてください。')


In [ ]:
# 4. 全編が存在する場合に実ファイルを検証
import imageio.v2 as imageio
VIDEO_NAME = 'PhysicalAI_5色ボール_坂・正面アーム左右仕分け_3セット_20261101.mp4'
video_path = WORK / VIDEO_NAME
if video_path.exists():
    reader = imageio.get_reader(video_path)
    metadata = reader.get_meta_data()
    count = reader.count_frames()
    reader.close()
    assert count == 2160
    assert tuple(metadata['size']) == (1280, 720)
    assert metadata['fps'] == 30
    assert abs(metadata['duration'] - 72.0) < 0.01
    print('確認しました。72秒・1280×720・30fps・2,160フレームです。')
    from IPython.display import Video
    display(Video(str(video_path), embed=False))
else:
    print('全編未生成のため、この検証はまだ実施していません。')


## 提出前の最終確認

**確認しました。**

提出用Notebook・Pythonソース・説明文・設定・動画を対象に、氏名・所属・メールアドレス、ローカル個人パス、アカウント識別子、認証情報が含まれないことを確認しました。Notebookの実行履歴・出力欄は空にし、作成者メタデータを持たせていません。

動画には人工的な装置・球・動作説明だけを描き、人物、実写画面、個人情報を表示しません。全15球の坂・接触・トレー到達の抽出フレーム、五色×3セットの一覧、MP4のメタデータを確認しました。全2,160フレームのUI文言はソースから生成される固定文言と色名・連番であることも点検しました。全フレームを人手で逐一目視した、とは主張しません。

フォント本体は提出物には含めません。実行時に取得したフォントやライセンス、環境ログを再提出する場合は、それらも別途確認してください。公開対象は提出専用フォルダのファイルだけです。
